# Catalog · Schema · Volume · Table DDLs — IntelliBI Sales Lakehouse

Single source of truth for all object provisioning.  Runs in **dev**, **qa**, **prd**.

* Existing entities (Customer, Sales, Discount, Invoice) — unchanged.
* **New entities (Region, Product, Store + Dimensions + Facts) point to AWS S3** at:
  `s3://saleslake-748005667258-eu-north-1-an/saleslake/<env>/delta_tables/<layer>/<entity>/`
* All duplicates from earlier revisions removed.


## 1. Catalogs, Schemas, Volumes

In [0]:
%sql
-- ===== Catalogs (one per env) =====
CREATE CATALOG IF NOT EXISTS saleslake_dev;
CREATE CATALOG IF NOT EXISTS saleslake_qa;
CREATE CATALOG IF NOT EXISTS saleslake_prd;

-- ===== Schemas (bronze / silver / gold per env) =====
CREATE SCHEMA IF NOT EXISTS saleslake_dev.bronze_dev;
CREATE SCHEMA IF NOT EXISTS saleslake_dev.silver_dev;
CREATE SCHEMA IF NOT EXISTS saleslake_dev.gold_dev;
CREATE SCHEMA IF NOT EXISTS saleslake_qa.bronze_qa;
CREATE SCHEMA IF NOT EXISTS saleslake_qa.silver_qa;
CREATE SCHEMA IF NOT EXISTS saleslake_qa.gold_qa;
CREATE SCHEMA IF NOT EXISTS saleslake_prd.bronze_prd;
CREATE SCHEMA IF NOT EXISTS saleslake_prd.silver_prd;
CREATE SCHEMA IF NOT EXISTS saleslake_prd.gold_prd;

-- ===== Volumes (source-file landing zone + archive) =====
CREATE VOLUME IF NOT EXISTS saleslake_dev.bronze_dev.vol_saleslake_src_files_dev;
CREATE VOLUME IF NOT EXISTS saleslake_qa.bronze_qa.vol_saleslake_src_files_qa;
CREATE VOLUME IF NOT EXISTS saleslake_prd.bronze_prd.vol_saleslake_src_files_prd;
CREATE VOLUME IF NOT EXISTS saleslake_dev.bronze_dev.vol_saleslake_archive_files_dev;
CREATE VOLUME IF NOT EXISTS saleslake_qa.bronze_qa.vol_saleslake_archive_files_qa;
CREATE VOLUME IF NOT EXISTS saleslake_prd.bronze_prd.vol_saleslake_archive_files_prd;


## 2. Bronze Layer
Existing tables (`rawCustomer`, `rawDiscount`, `rawSales`, `rawInvoice`) kept as-is.
New entities (`rawRegion`, `rawProduct`, `rawStore`) **point to AWS S3**.

In [0]:
%sql
-- ===== EXISTING: rawCustomer  (managed table) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.bronze_dev.rawCustomer (
    customer_id STRING, customer_name STRING, email STRING,
    phone STRING, address STRING, city STRING, state STRING,
    country STRING, zip_code STRING, segment STRING,
    ingest_ts TIMESTAMP
);

CREATE TABLE IF NOT EXISTS saleslake_qa.bronze_qa.rawCustomer (
    customer_id STRING, customer_name STRING, email STRING,
    phone STRING, address STRING, city STRING, state STRING,
    country STRING, zip_code STRING, segment STRING,
    ingest_ts TIMESTAMP
);

CREATE TABLE IF NOT EXISTS saleslake_prd.bronze_prd.rawCustomer (
    customer_id STRING, customer_name STRING, email STRING,
    phone STRING, address STRING, city STRING, state STRING,
    country STRING, zip_code STRING, segment STRING,
    ingest_ts TIMESTAMP
);

In [0]:
%sql
-- DROP TABLE saleslake_dev.bronze_dev.rawDiscount;
-- DROP TABLE saleslake_qa.bronze_qa.rawDiscount;
-- DROP TABLE saleslake_prd.bronze_prd.rawDiscount;
-- ===== EXISTING: rawDiscount  (managed table) =====
CREATE OR REPLACE TABLE  saleslake_dev.bronze_dev.rawDiscount (
    discount_id STRING, discount_code STRING, discount_name STRING,
    discount_type STRING, discount_value STRING,
    min_purchase_amount STRING, max_discount_amount STRING,
    valid_from STRING, valid_to STRING,
    applicable_segment STRING, applicable_category STRING,
    usage_limit_per_customer STRING, total_usage_limit STRING,
    is_active STRING, created_date STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/bronze/discount/';

CREATE OR REPLACE TABLE  saleslake_qa.bronze_qa.rawDiscount (
    discount_id STRING, discount_code STRING, discount_name STRING,
    discount_type STRING, discount_value STRING,
    min_purchase_amount STRING, max_discount_amount STRING,
    valid_from STRING, valid_to STRING,
    applicable_segment STRING, applicable_category STRING,
    usage_limit_per_customer STRING, total_usage_limit STRING,
    is_active STRING, created_date STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/bronze/discount/';

CREATE OR REPLACE TABLE  saleslake_prd.bronze_prd.rawDiscount (
    discount_id STRING, discount_code STRING, discount_name STRING,
    discount_type STRING, discount_value STRING,
    min_purchase_amount STRING, max_discount_amount STRING,
    valid_from STRING, valid_to STRING,
    applicable_segment STRING, applicable_category STRING,
    usage_limit_per_customer STRING, total_usage_limit STRING,
    is_active STRING, created_date STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/bronze/discount/';

In [0]:
%sql
-- ===== ENHANCED: rawSales  (managed table; expanded schema with new FKs) =====
CREATE OR REPLACE TABLE saleslake_dev.bronze_dev.rawSales (
    sale_id STRING, product_id STRING, store_id STRING,
    customer_id STRING, region_id STRING,
    quantity STRING, unit_price STRING, gross_amount STRING,
    discount_code STRING, discount_amount STRING,
    net_amount STRING, cost_amount STRING,
    sale_date STRING, payment_method STRING, channel STRING,
    ingest_ts TIMESTAMP
);

CREATE OR REPLACE TABLE saleslake_qa.bronze_qa.rawSales (
    sale_id STRING, product_id STRING, store_id STRING,
    customer_id STRING, region_id STRING,
    quantity STRING, unit_price STRING, gross_amount STRING,
    discount_code STRING, discount_amount STRING,
    net_amount STRING, cost_amount STRING,
    sale_date STRING, payment_method STRING, channel STRING,
    ingest_ts TIMESTAMP
);

CREATE OR REPLACE TABLE saleslake_prd.bronze_prd.rawSales (
    sale_id STRING, product_id STRING, store_id STRING,
    customer_id STRING, region_id STRING,
    quantity STRING, unit_price STRING, gross_amount STRING,
    discount_code STRING, discount_amount STRING,
    net_amount STRING, cost_amount STRING,
    sale_date STRING, payment_method STRING, channel STRING,
    ingest_ts TIMESTAMP
);

In [0]:
%sql
-- ===== EXISTING: rawInvoice  (external; already on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.bronze_dev.rawInvoice (
    invoice_id STRING, invoice_number STRING, customer_id STRING,
    invoice_date STRING, due_date STRING,
    subtotal_amount STRING, discount_code STRING, discount_amount STRING,
    tax_amount STRING, total_amount STRING,
    payment_status STRING, payment_method STRING, payment_date STRING,
    currency STRING, region STRING, store_id STRING, channel STRING,
    created_by STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/bronze/invoice/';

CREATE TABLE IF NOT EXISTS saleslake_qa.bronze_qa.rawInvoice (
    invoice_id STRING, invoice_number STRING, customer_id STRING,
    invoice_date STRING, due_date STRING,
    subtotal_amount STRING, discount_code STRING, discount_amount STRING,
    tax_amount STRING, total_amount STRING,
    payment_status STRING, payment_method STRING, payment_date STRING,
    currency STRING, region STRING, store_id STRING, channel STRING,
    created_by STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/bronze/invoice/';

CREATE TABLE IF NOT EXISTS saleslake_prd.bronze_prd.rawInvoice (
    invoice_id STRING, invoice_number STRING, customer_id STRING,
    invoice_date STRING, due_date STRING,
    subtotal_amount STRING, discount_code STRING, discount_amount STRING,
    tax_amount STRING, total_amount STRING,
    payment_status STRING, payment_method STRING, payment_date STRING,
    currency STRING, region STRING, store_id STRING, channel STRING,
    created_by STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/bronze/invoice/';

In [0]:
%sql
-- ===== NEW: rawRegion  (external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.bronze_dev.rawRegion (
    region_id STRING, region_code STRING, region_name STRING,
    country STRING, sub_region STRING, regional_manager STRING,
    created_date STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/bronze/region/';

CREATE TABLE IF NOT EXISTS saleslake_qa.bronze_qa.rawRegion (
    region_id STRING, region_code STRING, region_name STRING,
    country STRING, sub_region STRING, regional_manager STRING,
    created_date STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/bronze/region/';

CREATE TABLE IF NOT EXISTS saleslake_prd.bronze_prd.rawRegion (
    region_id STRING, region_code STRING, region_name STRING,
    country STRING, sub_region STRING, regional_manager STRING,
    created_date STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/bronze/region/';

In [0]:
%sql
-- ===== NEW: rawProduct  (external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.bronze_dev.rawProduct (
    product_id STRING, sku STRING, product_name STRING,
    category STRING, sub_category STRING, brand STRING,
    supplier STRING, unit_cost STRING, list_price STRING,
    launch_date STRING, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/bronze/product/';

CREATE TABLE IF NOT EXISTS saleslake_qa.bronze_qa.rawProduct (
    product_id STRING, sku STRING, product_name STRING,
    category STRING, sub_category STRING, brand STRING,
    supplier STRING, unit_cost STRING, list_price STRING,
    launch_date STRING, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/bronze/product/';

CREATE TABLE IF NOT EXISTS saleslake_prd.bronze_prd.rawProduct (
    product_id STRING, sku STRING, product_name STRING,
    category STRING, sub_category STRING, brand STRING,
    supplier STRING, unit_cost STRING, list_price STRING,
    launch_date STRING, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/bronze/product/';

In [0]:
%sql
drop table saleslake_qa.bronze_qa.rawmonthlysales;
-- ===== NEW: rawStore  (external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.bronze_dev.rawStore (
    store_id STRING, store_code STRING, store_name STRING,
    store_type STRING, address STRING, city STRING, state STRING,
    region_id STRING, manager_name STRING, opening_date STRING,
    square_feet STRING, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/bronze/store/';

CREATE TABLE IF NOT EXISTS saleslake_qa.bronze_qa.rawStore (
    store_id STRING, store_code STRING, store_name STRING,
    store_type STRING, address STRING, city STRING, state STRING,
    region_id STRING, manager_name STRING, opening_date STRING,
    square_feet STRING, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/bronze/store/';

CREATE TABLE IF NOT EXISTS saleslake_prd.bronze_prd.rawStore (
    store_id STRING, store_code STRING, store_name STRING,
    store_type STRING, address STRING, city STRING, state STRING,
    region_id STRING, manager_name STRING, opening_date STRING,
    square_feet STRING, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/bronze/store/';

## 3. Silver Layer
Existing tables unchanged. New silver tables for region/product/store/discount on AWS S3.

In [0]:
%sql
-- ===== ENHANCED: cleanedSales =====
CREATE OR REPLACE TABLE saleslake_dev.silver_dev.cleanedSales (
    sale_id BIGINT, product_id INT, store_id INT,
    customer_id INT, region_id INT,
    quantity INT, unit_price DECIMAL(18,2), gross_amount DECIMAL(18,2),
    discount_code STRING, discount_amount DECIMAL(18,2),
    net_amount DECIMAL(18,2), cost_amount DECIMAL(18,2),
    sale_date DATE, payment_method STRING, channel STRING,
    ingest_ts TIMESTAMP
)
PARTITIONED BY (sale_date);

CREATE OR REPLACE TABLE saleslake_qa.silver_qa.cleanedSales (
    sale_id BIGINT, product_id INT, store_id INT,
    customer_id INT, region_id INT,
    quantity INT, unit_price DECIMAL(18,2), gross_amount DECIMAL(18,2),
    discount_code STRING, discount_amount DECIMAL(18,2),
    net_amount DECIMAL(18,2), cost_amount DECIMAL(18,2),
    sale_date DATE, payment_method STRING, channel STRING,
    ingest_ts TIMESTAMP
)
PARTITIONED BY (sale_date);

CREATE OR REPLACE TABLE saleslake_prd.silver_prd.cleanedSales (
    sale_id BIGINT, product_id INT, store_id INT,
    customer_id INT, region_id INT,
    quantity INT, unit_price DECIMAL(18,2), gross_amount DECIMAL(18,2),
    discount_code STRING, discount_amount DECIMAL(18,2),
    net_amount DECIMAL(18,2), cost_amount DECIMAL(18,2),
    sale_date DATE, payment_method STRING, channel STRING,
    ingest_ts TIMESTAMP
)
PARTITIONED BY (sale_date);

In [0]:
%sql
-- ===== EXISTING: cleanedCustomer =====
CREATE TABLE IF NOT EXISTS saleslake_dev.silver_dev.cleanedCustomer (
    customer_id INT, customer_name STRING, email STRING, phone STRING,
    address STRING, city STRING, state STRING, country STRING,
    zip_code STRING, segment STRING,
    ingest_ts TIMESTAMP
);

CREATE TABLE IF NOT EXISTS saleslake_qa.silver_qa.cleanedCustomer (
    customer_id INT, customer_name STRING, email STRING, phone STRING,
    address STRING, city STRING, state STRING, country STRING,
    zip_code STRING, segment STRING,
    ingest_ts TIMESTAMP
);

CREATE TABLE IF NOT EXISTS saleslake_prd.silver_prd.cleanedCustomer (
    customer_id INT, customer_name STRING, email STRING, phone STRING,
    address STRING, city STRING, state STRING, country STRING,
    zip_code STRING, segment STRING,
    ingest_ts TIMESTAMP
);

In [0]:
%sql
-- ===== EXISTING: cleanedInvoice  (external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.silver_dev.cleanedInvoice (
    invoice_id STRING NOT NULL, invoice_number STRING,
    customer_id BIGINT, invoice_date DATE, due_date DATE,
    subtotal_amount DECIMAL(18,2), discount_code STRING,
    discount_amount DECIMAL(18,2), tax_amount DECIMAL(18,2),
    total_amount DECIMAL(18,2),
    payment_status STRING, payment_method STRING, payment_date DATE,
    currency STRING, region STRING, store_id STRING, channel STRING,
    created_by STRING,
    ingest_ts TIMESTAMP NOT NULL
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/silver/invoice/';

CREATE TABLE IF NOT EXISTS saleslake_qa.silver_qa.cleanedInvoice (
    invoice_id STRING NOT NULL, invoice_number STRING,
    customer_id BIGINT, invoice_date DATE, due_date DATE,
    subtotal_amount DECIMAL(18,2), discount_code STRING,
    discount_amount DECIMAL(18,2), tax_amount DECIMAL(18,2),
    total_amount DECIMAL(18,2),
    payment_status STRING, payment_method STRING, payment_date DATE,
    currency STRING, region STRING, store_id STRING, channel STRING,
    created_by STRING,
    ingest_ts TIMESTAMP NOT NULL
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/silver/invoice/';

CREATE TABLE IF NOT EXISTS saleslake_prd.silver_prd.cleanedInvoice (
    invoice_id STRING NOT NULL, invoice_number STRING,
    customer_id BIGINT, invoice_date DATE, due_date DATE,
    subtotal_amount DECIMAL(18,2), discount_code STRING,
    discount_amount DECIMAL(18,2), tax_amount DECIMAL(18,2),
    total_amount DECIMAL(18,2),
    payment_status STRING, payment_method STRING, payment_date DATE,
    currency STRING, region STRING, store_id STRING, channel STRING,
    created_by STRING,
    ingest_ts TIMESTAMP NOT NULL
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/silver/invoice/';

In [0]:
%sql
-- ===== NEW: cleanedRegion  (external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.silver_dev.cleanedRegion (
    region_id INT, region_code STRING, region_name STRING,
    country STRING, sub_region STRING, regional_manager STRING,
    created_date DATE,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/silver/region/';

CREATE TABLE IF NOT EXISTS saleslake_qa.silver_qa.cleanedRegion (
    region_id INT, region_code STRING, region_name STRING,
    country STRING, sub_region STRING, regional_manager STRING,
    created_date DATE,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/silver/region/';

CREATE TABLE IF NOT EXISTS saleslake_prd.silver_prd.cleanedRegion (
    region_id INT, region_code STRING, region_name STRING,
    country STRING, sub_region STRING, regional_manager STRING,
    created_date DATE,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/silver/region/';

In [0]:
%sql
-- ===== NEW: cleanedProduct  (external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.silver_dev.cleanedProduct (
    product_id INT, sku STRING, product_name STRING,
    category STRING, sub_category STRING, brand STRING,
    supplier STRING, unit_cost DECIMAL(18,2), list_price DECIMAL(18,2),
    launch_date DATE, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/silver/product/';

CREATE TABLE IF NOT EXISTS saleslake_qa.silver_qa.cleanedProduct (
    product_id INT, sku STRING, product_name STRING,
    category STRING, sub_category STRING, brand STRING,
    supplier STRING, unit_cost DECIMAL(18,2), list_price DECIMAL(18,2),
    launch_date DATE, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/silver/product/';

CREATE TABLE IF NOT EXISTS saleslake_prd.silver_prd.cleanedProduct (
    product_id INT, sku STRING, product_name STRING,
    category STRING, sub_category STRING, brand STRING,
    supplier STRING, unit_cost DECIMAL(18,2), list_price DECIMAL(18,2),
    launch_date DATE, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/silver/product/';

In [0]:
%sql
-- ===== NEW: cleanedStore  (external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.silver_dev.cleanedStore (
    store_id INT, store_code STRING, store_name STRING,
    store_type STRING, address STRING, city STRING, state STRING,
    region_id INT, manager_name STRING, opening_date DATE,
    square_feet INT, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/silver/store/';

CREATE TABLE IF NOT EXISTS saleslake_qa.silver_qa.cleanedStore (
    store_id INT, store_code STRING, store_name STRING,
    store_type STRING, address STRING, city STRING, state STRING,
    region_id INT, manager_name STRING, opening_date DATE,
    square_feet INT, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/silver/store/';

CREATE TABLE IF NOT EXISTS saleslake_prd.silver_prd.cleanedStore (
    store_id INT, store_code STRING, store_name STRING,
    store_type STRING, address STRING, city STRING, state STRING,
    region_id INT, manager_name STRING, opening_date DATE,
    square_feet INT, status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/silver/store/';

In [0]:
%sql
-- ===== NEW: cleanedDiscount  (external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.silver_dev.cleanedDiscount (
    discount_id INT, discount_code STRING, discount_name STRING,
    discount_type STRING, discount_value DECIMAL(10,2),
    min_purchase_amount DECIMAL(18,2), max_discount_amount DECIMAL(18,2),
    valid_from DATE, valid_to DATE,
    applicable_segment STRING, applicable_category STRING,
    usage_limit_per_customer INT, total_usage_limit INT,
    is_active STRING, created_date DATE,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/silver/discount/';

CREATE TABLE IF NOT EXISTS saleslake_qa.silver_qa.cleanedDiscount (
    discount_id INT, discount_code STRING, discount_name STRING,
    discount_type STRING, discount_value DECIMAL(10,2),
    min_purchase_amount DECIMAL(18,2), max_discount_amount DECIMAL(18,2),
    valid_from DATE, valid_to DATE,
    applicable_segment STRING, applicable_category STRING,
    usage_limit_per_customer INT, total_usage_limit INT,
    is_active STRING, created_date DATE,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/silver/discount/';

CREATE TABLE IF NOT EXISTS saleslake_prd.silver_prd.cleanedDiscount (
    discount_id INT, discount_code STRING, discount_name STRING,
    discount_type STRING, discount_value DECIMAL(10,2),
    min_purchase_amount DECIMAL(18,2), max_discount_amount DECIMAL(18,2),
    valid_from DATE, valid_to DATE,
    applicable_segment STRING, applicable_category STRING,
    usage_limit_per_customer INT, total_usage_limit INT,
    is_active STRING, created_date DATE,
    ingest_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/silver/discount/';

## 4. Gold Layer — Dimensions, Facts, Date
Existing `refinedSales`, `refinedCustomer`, `refinedInvoice` kept.
All new dims and facts **point to AWS S3** and use surrogate identity keys.

In [0]:
%sql
-- ===== EXISTING: refinedSales  (SCD1, existing schema) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.gold_dev.refinedSales (
    sale_id INT, product STRING, category STRING,
    quantity INT, price DOUBLE, sale_date DATE, region STRING,
    initial_load_ts TIMESTAMP, last_updt_ts TIMESTAMP
);

CREATE TABLE IF NOT EXISTS saleslake_qa.gold_qa.refinedSales (
    sale_id INT, product STRING, category STRING,
    quantity INT, price DOUBLE, sale_date DATE, region STRING,
    initial_load_ts TIMESTAMP, last_updt_ts TIMESTAMP
);

CREATE TABLE IF NOT EXISTS saleslake_prd.gold_prd.refinedSales (
    sale_id INT, product STRING, category STRING,
    quantity INT, price DOUBLE, sale_date DATE, region STRING,
    initial_load_ts TIMESTAMP, last_updt_ts TIMESTAMP
);

In [0]:
%sql
-- ===== EXISTING: refinedCustomer  (SCD2, existing) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.gold_dev.refinedCustomer (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id INT, customer_name STRING, email STRING, phone STRING,
    address STRING, city STRING, state STRING, country STRING,
    zip_code STRING, segment STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
);

CREATE TABLE IF NOT EXISTS saleslake_qa.gold_qa.refinedCustomer (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id INT, customer_name STRING, email STRING, phone STRING,
    address STRING, city STRING, state STRING, country STRING,
    zip_code STRING, segment STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
);

CREATE TABLE IF NOT EXISTS saleslake_prd.gold_prd.refinedCustomer (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id INT, customer_name STRING, email STRING, phone STRING,
    address STRING, city STRING, state STRING, country STRING,
    zip_code STRING, segment STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
);

In [0]:
%sql
-- ===== EXISTING: refinedInvoice  (external on AWS S3) =====
-- DROP TABLE saleslake_dev.gold_dev.refinedInvoice;
CREATE TABLE IF NOT EXISTS saleslake_dev.gold_dev.dim_invoice (
    invoice_id STRING NOT NULL, invoice_number STRING,
    customer_id BIGINT, invoice_date DATE, due_date DATE,
    subtotal_amount DECIMAL(18,2), discount_code STRING,
    discount_amount DECIMAL(18,2), tax_amount DECIMAL(18,2),
    total_amount DECIMAL(18,2),
    payment_status STRING, payment_method STRING, payment_date DATE,
    currency STRING, region STRING, store_id STRING, channel STRING,
    created_by STRING,
    initial_load_ts TIMESTAMP NOT NULL, last_updt_ts TIMESTAMP NOT NULL
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/gold/dim_invoice/';


CREATE TABLE IF NOT EXISTS saleslake_qa.gold_qa.dim_invoice (
    invoice_id STRING NOT NULL, invoice_number STRING,
    customer_id BIGINT, invoice_date DATE, due_date DATE,
    subtotal_amount DECIMAL(18,2), discount_code STRING,
    discount_amount DECIMAL(18,2), tax_amount DECIMAL(18,2),
    total_amount DECIMAL(18,2),
    payment_status STRING, payment_method STRING, payment_date DATE,
    currency STRING, region STRING, store_id STRING, channel STRING,
    created_by STRING,
    initial_load_ts TIMESTAMP NOT NULL, last_updt_ts TIMESTAMP NOT NULL
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/gold/dim_invoice/';

CREATE TABLE IF NOT EXISTS saleslake_prd.gold_prd.refinedInvoice (
    invoice_id STRING NOT NULL, invoice_number STRING,
    customer_id BIGINT, invoice_date DATE, due_date DATE,
    subtotal_amount DECIMAL(18,2), discount_code STRING,
    discount_amount DECIMAL(18,2), tax_amount DECIMAL(18,2),
    total_amount DECIMAL(18,2),
    payment_status STRING, payment_method STRING, payment_date DATE,
    currency STRING, region STRING, store_id STRING, channel STRING,
    created_by STRING,
    initial_load_ts TIMESTAMP NOT NULL, last_updt_ts TIMESTAMP NOT NULL
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/gold/dim_invoice/';

In [0]:
%sql
-- ===== NEW: dim_region  (SCD Type 1, external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.gold_dev.dim_region (
    region_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    region_id INT, region_code STRING, region_name STRING,
    country STRING, sub_region STRING, regional_manager STRING,
    initial_load_ts TIMESTAMP, last_updt_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/gold/dim_region/';

CREATE TABLE IF NOT EXISTS saleslake_qa.gold_qa.dim_region (
    region_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    region_id INT, region_code STRING, region_name STRING,
    country STRING, sub_region STRING, regional_manager STRING,
    initial_load_ts TIMESTAMP, last_updt_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/gold/dim_region/';

CREATE TABLE IF NOT EXISTS saleslake_prd.gold_prd.dim_region (
    region_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    region_id INT, region_code STRING, region_name STRING,
    country STRING, sub_region STRING, regional_manager STRING,
    initial_load_ts TIMESTAMP, last_updt_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/gold/dim_region/';

In [0]:
%sql
-- ===== NEW: dim_product  (SCD Type 2, external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.gold_dev.dim_product (
    product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id INT, sku STRING, product_name STRING,
    category STRING, sub_category STRING, brand STRING,
    supplier STRING, unit_cost DECIMAL(18,2), list_price DECIMAL(18,2),
    launch_date DATE, status STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/gold/dim_product/';

CREATE TABLE IF NOT EXISTS saleslake_qa.gold_qa.dim_product (
    product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id INT, sku STRING, product_name STRING,
    category STRING, sub_category STRING, brand STRING,
    supplier STRING, unit_cost DECIMAL(18,2), list_price DECIMAL(18,2),
    launch_date DATE, status STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/gold/dim_product/';

CREATE TABLE IF NOT EXISTS saleslake_prd.gold_prd.dim_product (
    product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id INT, sku STRING, product_name STRING,
    category STRING, sub_category STRING, brand STRING,
    supplier STRING, unit_cost DECIMAL(18,2), list_price DECIMAL(18,2),
    launch_date DATE, status STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/gold/dim_product/';

In [0]:
%sql
-- ===== NEW: dim_store  (SCD Type 2, external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.gold_dev.dim_store (
    store_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    store_id INT, store_code STRING, store_name STRING,
    store_type STRING, address STRING, city STRING, state STRING,
    region_id INT, manager_name STRING, opening_date DATE,
    square_feet INT, status STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/gold/dim_store/';

CREATE TABLE IF NOT EXISTS saleslake_qa.gold_qa.dim_store (
    store_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    store_id INT, store_code STRING, store_name STRING,
    store_type STRING, address STRING, city STRING, state STRING,
    region_id INT, manager_name STRING, opening_date DATE,
    square_feet INT, status STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/gold/dim_store/';

CREATE TABLE IF NOT EXISTS saleslake_prd.gold_prd.dim_store (
    store_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    store_id INT, store_code STRING, store_name STRING,
    store_type STRING, address STRING, city STRING, state STRING,
    region_id INT, manager_name STRING, opening_date DATE,
    square_feet INT, status STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/gold/dim_store/';

In [0]:
%sql
-- ===== NEW: dim_discount  (SCD Type 2, external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.gold_dev.dim_discount (
    discount_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    discount_id INT, discount_code STRING, discount_name STRING,
    discount_type STRING, discount_value DECIMAL(10,2),
    min_purchase_amount DECIMAL(18,2), max_discount_amount DECIMAL(18,2),
    valid_from DATE, valid_to DATE,
    applicable_segment STRING, applicable_category STRING,
    usage_limit_per_customer INT, total_usage_limit INT, status STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/gold/dim_discount/';

CREATE TABLE IF NOT EXISTS saleslake_qa.gold_qa.dim_discount (
    discount_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    discount_id INT, discount_code STRING, discount_name STRING,
    discount_type STRING, discount_value DECIMAL(10,2),
    min_purchase_amount DECIMAL(18,2), max_discount_amount DECIMAL(18,2),
    valid_from DATE, valid_to DATE,
    applicable_segment STRING, applicable_category STRING,
    usage_limit_per_customer INT, total_usage_limit INT, status STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/gold/dim_discount/';

CREATE TABLE IF NOT EXISTS saleslake_prd.gold_prd.dim_discount (
    discount_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    discount_id INT, discount_code STRING, discount_name STRING,
    discount_type STRING, discount_value DECIMAL(10,2),
    min_purchase_amount DECIMAL(18,2), max_discount_amount DECIMAL(18,2),
    valid_from DATE, valid_to DATE,
    applicable_segment STRING, applicable_category STRING,
    usage_limit_per_customer INT, total_usage_limit INT, status STRING,
    is_active STRING, rec_version INT,
    start_effective_ts TIMESTAMP, end_effective_ts TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/gold/dim_discount/';

In [0]:
%sql
-- ===== NEW: dim_date  (static date dim, external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.gold_dev.dim_date (
    date_sk INT, full_date DATE, day INT, month INT, month_name STRING,
    quarter INT, year INT, day_of_week INT, day_name STRING,
    week_of_year INT, is_weekend BOOLEAN, is_month_end BOOLEAN,
    fiscal_year INT, fiscal_quarter INT
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/gold/dim_date/';

CREATE TABLE IF NOT EXISTS saleslake_qa.gold_qa.dim_date (
    date_sk INT, full_date DATE, day INT, month INT, month_name STRING,
    quarter INT, year INT, day_of_week INT, day_name STRING,
    week_of_year INT, is_weekend BOOLEAN, is_month_end BOOLEAN,
    fiscal_year INT, fiscal_quarter INT
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/gold/dim_date/';

CREATE TABLE IF NOT EXISTS saleslake_prd.gold_prd.dim_date (
    date_sk INT, full_date DATE, day INT, month INT, month_name STRING,
    quarter INT, year INT, day_of_week INT, day_name STRING,
    week_of_year INT, is_weekend BOOLEAN, is_month_end BOOLEAN,
    fiscal_year INT, fiscal_quarter INT
)
USING DELTA
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/gold/dim_date/';

In [0]:
%sql
-- ===== NEW: fact_sales  (Star schema fact, external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.gold_dev.fact_sales (
    sale_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    sale_id BIGINT,
    date_sk INT, customer_sk BIGINT, product_sk BIGINT,
    store_sk BIGINT, region_sk BIGINT, discount_sk BIGINT,
    sale_date DATE,
    quantity INT, unit_price DECIMAL(18,2),
    gross_amount DECIMAL(18,2), discount_amount DECIMAL(18,2),
    net_amount DECIMAL(18,2), cost_amount DECIMAL(18,2),
    profit_amount DECIMAL(18,2), profit_margin_pct DECIMAL(8,4),
    payment_method STRING, channel STRING,
    load_ts TIMESTAMP
)
USING DELTA
PARTITIONED BY (sale_date)
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/gold/fact_sales/';

CREATE TABLE IF NOT EXISTS saleslake_qa.gold_qa.fact_sales (
    sale_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    sale_id BIGINT,
    date_sk INT, customer_sk BIGINT, product_sk BIGINT,
    store_sk BIGINT, region_sk BIGINT, discount_sk BIGINT,
    sale_date DATE,
    quantity INT, unit_price DECIMAL(18,2),
    gross_amount DECIMAL(18,2), discount_amount DECIMAL(18,2),
    net_amount DECIMAL(18,2), cost_amount DECIMAL(18,2),
    profit_amount DECIMAL(18,2), profit_margin_pct DECIMAL(8,4),
    payment_method STRING, channel STRING,
    load_ts TIMESTAMP
)
USING DELTA
PARTITIONED BY (sale_date)
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/gold/fact_sales/';

CREATE TABLE IF NOT EXISTS saleslake_prd.gold_prd.fact_sales (
    sale_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    sale_id BIGINT,
    date_sk INT, customer_sk BIGINT, product_sk BIGINT,
    store_sk BIGINT, region_sk BIGINT, discount_sk BIGINT,
    sale_date DATE,
    quantity INT, unit_price DECIMAL(18,2),
    gross_amount DECIMAL(18,2), discount_amount DECIMAL(18,2),
    net_amount DECIMAL(18,2), cost_amount DECIMAL(18,2),
    profit_amount DECIMAL(18,2), profit_margin_pct DECIMAL(8,4),
    payment_method STRING, channel STRING,
    load_ts TIMESTAMP
)
USING DELTA
PARTITIONED BY (sale_date)
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/gold/fact_sales/';

In [0]:
%sql
-- ===== NEW: fact_invoice  (Star schema fact, external on AWS S3) =====
CREATE TABLE IF NOT EXISTS saleslake_dev.gold_dev.fact_invoice (
    invoice_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    invoice_id STRING, invoice_number STRING,
    customer_sk BIGINT, store_sk BIGINT,
    region_sk BIGINT, discount_sk BIGINT,
    invoice_date_sk INT, due_date_sk INT, payment_date_sk INT,
    invoice_date DATE, due_date DATE, payment_date DATE,
    subtotal_amount DECIMAL(18,2), discount_amount DECIMAL(18,2),
    tax_amount DECIMAL(18,2), total_amount DECIMAL(18,2),
    payment_status STRING, payment_method STRING, channel STRING,
    days_to_pay INT, is_overdue BOOLEAN,
    load_ts TIMESTAMP
)
USING DELTA
PARTITIONED BY (invoice_date)
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/dev/delta_tables/gold/fact_invoice/';

CREATE TABLE IF NOT EXISTS saleslake_qa.gold_qa.fact_invoice (
    invoice_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    invoice_id STRING, invoice_number STRING,
    customer_sk BIGINT, store_sk BIGINT,
    region_sk BIGINT, discount_sk BIGINT,
    invoice_date_sk INT, due_date_sk INT, payment_date_sk INT,
    invoice_date DATE, due_date DATE, payment_date DATE,
    subtotal_amount DECIMAL(18,2), discount_amount DECIMAL(18,2),
    tax_amount DECIMAL(18,2), total_amount DECIMAL(18,2),
    payment_status STRING, payment_method STRING, channel STRING,
    days_to_pay INT, is_overdue BOOLEAN,
    load_ts TIMESTAMP
)
USING DELTA
PARTITIONED BY (invoice_date)
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/qa/delta_tables/gold/fact_invoice/';

CREATE TABLE IF NOT EXISTS saleslake_prd.gold_prd.fact_invoice (
    invoice_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    invoice_id STRING, invoice_number STRING,
    customer_sk BIGINT, store_sk BIGINT,
    region_sk BIGINT, discount_sk BIGINT,
    invoice_date_sk INT, due_date_sk INT, payment_date_sk INT,
    invoice_date DATE, due_date DATE, payment_date DATE,
    subtotal_amount DECIMAL(18,2), discount_amount DECIMAL(18,2),
    tax_amount DECIMAL(18,2), total_amount DECIMAL(18,2),
    payment_status STRING, payment_method STRING, channel STRING,
    days_to_pay INT, is_overdue BOOLEAN,
    load_ts TIMESTAMP
)
USING DELTA
PARTITIONED BY (invoice_date)
LOCATION 's3://saleslake-748005667258-eu-north-1-an/saleslake/prd/delta_tables/gold/fact_invoice/';

---
**End of DDL.**  After running this notebook every layer is provisioned;
the ingestion and SCD notebooks will then populate them.